# 🧠 استودیو جامع پردازش زبان طبیعی (Multilingual NLP Studio)
### پوشش کامل سرفصل‌های جلسات ۱ تا ۴: از متن خام چالش‌برانگیز تا مدل‌های کلاسیک یادگیری ماشین و Transformer

در این نوت‌بوک اجرایی، تمام مراحل یک سیستم واقعی پردازش زبان طبیعی روی یک پیکره داده سنگین و چندزبانه (فارسی، نویسه‌های عربی و انگلیسی) پیاده‌سازی و مقایسه شده است:
1. **گام ۱ — متن خام (Raw Text)**: بارگذاری اسناد طولانی حاوی نویز HTML، کاراکترهای نامرئی یونیکد، حروف عربی (`ي`, `ك`, `ة`, `أ`)، نیم‌فاصله‌های اشتباه، محاوره، ایموجی، لینک/ایمیل/شماره تماس آزمایشی و جملات منفی.
2. **گام ۲ — پاک‌سازی (Text Cleaning)**: حذف نویزها، جایگزینی هوشمند `<URL>`، `<EMAIL>`، `<USER>`، `<PHONE>`، `<NUM>` و یکدست‌سازی علائم نگارشی.
3. **گام ۳ — نرمال‌سازی ۴ سطحی (Normalization)**: ترکیب `Unicode NFKC` با جدول نگاشت عربی به فارسی، حذف اعراب و کشیده، اصلاح قواعد نیم‌فاصله (`ZWNJ`) و تبدیل محاوره به رسمی.
4. **گام ۴ — توکنایزیشن (Tokenization)**: مقایسه جمله‌بندی امن (بدون شکستن `Dr. Smith`، `U.S.A.` و `3.14`) و مقایسه روش‌های `whitespace`، `regex_smart` و `subword_bpe_demo`.
5. **گام ۵ — حذف ایست‌واژه‌ها (Stopwords Removal)**: مقایسه کامل **۵ روش جزوه** (`Dictionary`، `Statistical IDF`، `Domain-Specific`، `ML Classifier` و `Hybrid`) + نمایش عملی **تله حذف کلمات نفی** در تحلیل احساسات.
6. **گام ۶ — ریشه‌یابی (Stemming vs Lemmatization)**: مقایسه `Porter`، `Snowball` و `Lancaster` با `Lemmatization` دقیق (نقش دستوری POS و بن افعال فارسی) و بررسی کاهش اندازه واژگان (`Vocabulary Size` و `TTR`).
7. **گام ۷ — نمایش برداری و موتور جستجوی کسینوسی (BoW vs TF-IDF, Sparsity & Fuzzy Fallback)**: مقایسه `CountVectorizer` و `TfidfVectorizer` هموارشده با `n-gram (1,2)`، بررسی صرفه‌جویی حافظه (`Dense` vs `Sparse CSR`) و جستجوی دقیق + تطابق تقریبی/املایی.
8. **گام ۸ — مقایسه ۶ خانواده مدل کلاسیک و مدرن**: ارزیابی کنار هم **Rule-Based Lexicon**، **Multinomial Naive Bayes**، **Logistic Regression**، **Linear SVM**، **Random Forest Ensemble** و **Transformer (Self-Attention)** با معیارهای `Accuracy`، `Precision`، `Recall`، `F1-Score` و `Confusion Matrix`.

In [1]:
# Import kardan tamame module haye proje az pooshe src
import copy
import json
from src.config import PIPELINE_CONFIG
from src.cleaner import TextCleaner
from src.normalizer import TextNormalizer
from src.tokenizer import TextTokenizer
from src.stopwords_handler import StopwordsRemover
from src.stemmer_lemmatizer import MorphologyProcessor
from src.vectorizer import TextVectorizerEngine
from src.models_comparator import NLPModelsBenchmark
from src.pipeline import MultilingualNLPPipeline

print("Tamam module haye NLP Studio ba movafaghiat load shodand!")

Tamam module haye NLP Studio ba movafaghiat load shodand!


## گام ۱: بارگذاری پیکره داده سنگین و چالش‌برانگیز (Raw Multilingual Corpus)
ابتدا اسناد طولانی و چندلایه را از فایل `data/sample_texts.txt` می‌خوانیم تا تمام چالش‌های متنی فارسی، عربی و انگلیسی را در یک جا آزمایش کنیم.

In [2]:
with open("data/sample_texts.txt", "r", encoding="utf-8") as f:
    raw_docs = [line.strip() for line in f if line.strip()]

print(f"Tedad kol asnad sangin: {len(raw_docs)}")
print(f"Majmoo character haye kham: {sum(len(d) for d in raw_docs)}\n")
for idx, doc in enumerate(raw_docs, 1):
    print(f"[Doc #{idx}] (Len={len(doc)} chars): {doc[:130]}...")

Tedad kol asnad sangin: 7
Majmoo character haye kham: 4027

[Doc #1] (Len=817 chars): <article class='news-body'><h1>گزارش جامع تحول پردازش زبان طبیعی (NLP) در سال ۱۴۰۵ و ۲۰۲۶</h1>&nbsp;سلاااام به همه پژوهشگران و دان...
[Doc #2] (Len=486 chars): <div class='review-card negative'><p>تجربه خرید و بررسی فنی سرور پردازشی و مدل زبانی محلی:</p>متأسفانه کیفیت این محصول اصلاً خوب ن...
[Doc #3] (Len=462 chars): <div class='review-card positive'><p>ارزیابی عملکرد پایپ‌لاین جدید بازیابی دانش (RAG) و موتور جستجوی برداری:</p>این سیستم جدید پرد...
[Doc #4] (Len=653 chars): In 2026, Dr. Smith, Prof. Johnson, and their research team at U.S.A. were studying complex multilingual corpora (version 3.14 and ...
[Doc #5] (Len=738 chars): تحلیل عمیق چالش‌های صرفی، واژگانی و آماری در زبان فارسی و عربی: در پردازش زبان طبیعی فارسی، یک واژه می‌تواند ده‌ها شکل نوشتاری و ص...
[Doc #6] (Len=360 chars): گزارش پرونده حقوقی و مطالعه بالینی (تست ایست‌واژه‌های مبتنی بر دامنه - Domain-Specific Stopwords): طبق ماده ۱۴ قان

## گام ۲ و ۳: کالبدشکافی دقیق پاک‌سازی (Cleaning) و نرمال‌سازی ۴ سطحی (Normalization)
در این سلول، سند اول را مرحله‌به‌مرحله بررسی می‌کنیم تا تأثیر حذف تگ HTML، نویسه‌های نامرئی، یکسان‌سازی حروف عربی به فارسی و اصلاح نیم‌فاصله‌ها دیده شود.

In [3]:
cfg = copy.deepcopy(PIPELINE_CONFIG)
cleaner = TextCleaner(cfg)
normalizer = TextNormalizer(cfg)

heavy_doc_1 = raw_docs[0]
cleaned_1, clean_audit_1 = cleaner.clean(heavy_doc_1)
norm_1, norm_audit_1 = normalizer.normalize(cleaned_1)

print("--- 1. RAW TEXT ---")
print(heavy_doc_1)
print("\n--- 2. CLEANED TEXT ---")
print(cleaned_1)
print("Cleaning Audit Steps:", clean_audit_1["steps_applied"])
print("\n--- 3. NORMALIZED TEXT (4-LEVEL) ---")
print(norm_1)
print("Normalization Audit Changes:", norm_audit_1["changes"])

--- 1. RAW TEXT ---
<article class='news-body'><h1>گزارش جامع تحول پردازش زبان طبیعی (NLP) در سال ۱۴۰۵ و ۲۰۲۶</h1>&nbsp;سلاااام به همه پژوهشگران و دانشجویانِ علاقه‌مند به #هوش_مصنوعی، #پردازش_زبان_طبیعی و #یادگیری_عمیق!! أيا تا به حال به این موضوع فکر کرده‌اید که چرا كتا‌ب‌هاي قدیمیِ فارسی و مقالات علمیِ عربی در سیستم‌های جستجو به درستی بازیابی نمی‌شوند؟ وقتی یک کاربر عبارت «كتاب‌هاي هوش مصنوعي» را با حرف «ك» و «ي» عربی جستجو می‌کند، یا افعالی مانند «می‌روم»، «میروم»، «می‌خواهم»، «میخواهم»، «نمیدونم» و «بزرگترین» را با فاصله یا بدون نیم‌فاصله می‌نویسد، موتورهای جستجوی ساده دچار خطا می‌شوند! 😊🚀 برای دانلود مجموعه داده‌های استاندارد و مستندات فنی به آدرس https://nlp-research-lab.ir/datasets/v3.14 یا ایمیل contact@nlp-research-lab.ir مراجعه کنید و یا با پشتیبانی @nlp_support و شماره تماس 09120000000 در ارتباط باشید.​‎﻿</article>

--- 2. CLEANED TEXT ---
گزارش جامع تحول پردازش زبان طبیعی (NLP) در سال <NUM> و <NUM> سلاااام به همه پژوهشگران و دانشجویانِ علاقه‌مند به هوش مصنوعی، پردازش زبان ط

## گام ۴: مقایسه روش‌های توکنایزیشن (Whitespace vs Regex Smart vs Subword BPE)
آزمایش جمله‌بندی هوشمند روی متن انگلیسی دارای مخفف (`Dr. Smith`، `Prof. Johnson`، `U.S.A.`) و اعداد اعشاری (`3.14`، `2.71`) و مقایسه سه روش توکنایزیشن.

In [4]:
en_heavy_doc = raw_docs[3]
c_en, _ = cleaner.clean(en_heavy_doc)
n_en, _ = normalizer.normalize(c_en)

tok_engine = TextTokenizer(cfg)
sents_en = tok_engine.split_sentences(n_en)
print(f"Tedad jomle haye tashkhis dade shode: {len(sents_en)}")
for i, s in enumerate(sents_en, 1):
    print(f"  Sentence {i}: {s}")

print("\n--- Moghayese Ravesh haye Tokenization ---")
for t_method in ["whitespace", "regex_smart", "subword_bpe_demo"]:
    cfg_t = copy.deepcopy(PIPELINE_CONFIG)
    cfg_t["tokenization"]["method"] = t_method
    _, toks, t_aud = TextTokenizer(cfg_t).tokenize(n_en)
    print(f"{t_method:<18} (Count={len(toks)}): {toks[:10]}")
    if t_method == "regex_smart":
        print("   -> Token IDs (10 aval):", t_aud["token_ids_sample"][:10])
        print("   -> Padded IDs (Len 16):", t_aud["padded_sequence_len_16"]["input_ids"])
        print("   -> Attention Mask     :", t_aud["padded_sequence_len_16"]["attention_mask"])
        print("   -> One-Hot Vector Row 1:", t_aud["one_hot_preview"]["rows"][0])

Tedad jomle haye tashkhis dade shode: 5
  Sentence 1: in <num>, dr. smith, prof. johnson, and their research team at u
  Sentence 2: s
  Sentence 3: a. were studying complex multilingual corpora (version <num> and <num> ), running high-speed nlp pipelines, and analyzing why children's books, mice experiments, and messy web-scraped html documents were not producing better results
  Sentence 4: they discovered that unhappiness with preprocessing pipelines often happens when naive tokenization splits "do not" incorrectly or removes critical negations like "not good" and "never works", turning negative customer reviews into false positives
  Sentence 5: check <url> or contact <email> for full benchmark datasets

--- Moghayese Ravesh haye Tokenization ---
whitespace         (Count=85): ['in', '<num>,', 'dr.', 'smith,', 'prof.', 'johnson,', 'and', 'their', 'research', 'team']
regex_smart        (Count=89): ['in', '<num>', 'dr', 'smith', 'prof', 'johnson', 'and', 'their', 'research', 'team']


## گام ۵: مقایسه هر ۵ روش حذف ایست‌واژه‌ها + تست حیاتی کلمات نفی (Negation Guard)
مقایسه روش‌های `dictionary`، `statistical_idf`، `domain_specific`، `ml_classifier` و `hybrid` و بررسی نقش محافظ کلمات نفی.

In [5]:
domain_doc = raw_docs[5]
c_dom, _ = cleaner.clean(domain_doc)
n_dom, _ = normalizer.normalize(c_dom)
_, toks_dom, _ = tok_engine.tokenize(n_dom)

all_pre_toks = [tok_engine.tokenize(normalizer.normalize(cleaner.clean(d)[0])[0])[1] for d in raw_docs]
sw_base = StopwordsRemover(cfg)
stat_stops = sw_base.compute_statistical_stopwords(all_pre_toks)
ml_stops, ml_evidence = sw_base.detect_ml_stopwords(all_pre_toks)

print("Top kalame haye tashkhis dade shode tavasot ML Stopword Classifier:")
for item in ml_evidence[:5]:
    print(" ", item)

print("\n--- Moghayese 5 Ravesh Stopwords rooye Doc #6 ---")
for sw_method in ["dictionary", "statistical_idf", "domain_specific", "ml_classifier", "hybrid"]:
    cfg_s = copy.deepcopy(PIPELINE_CONFIG)
    cfg_s["stopwords"]["method"] = sw_method
    cfg_s["stopwords"]["domain"] = "legal"
    filt, aud = StopwordsRemover(cfg_s).filter_tokens(toks_dom, stat_stops, ml_stops)
    print(f"{sw_method:<16} -> Baghi-mande: {len(filt):2d} | Hazf shode: {aud['removed_tokens']}")

neg_doc = raw_docs[1]
_, toks_neg, _ = tok_engine.tokenize(normalizer.normalize(cleaner.clean(neg_doc)[0])[0])

cfg_keep = copy.deepcopy(PIPELINE_CONFIG)
cfg_keep["stopwords"]["keep_negations"] = True
kept_neg, _ = StopwordsRemover(cfg_keep).filter_tokens(toks_neg)

cfg_drop = copy.deepcopy(PIPELINE_CONFIG)
cfg_drop["stopwords"]["keep_negations"] = False
dropped_neg, _ = StopwordsRemover(cfg_drop).filter_tokens(toks_neg)

print("\n--- Negation Guard Test (Doc #2) ---")
print("Ba hefz nafi (Dorost) :", [w for w in kept_neg if w in ('خوب', 'نبود', 'پاسخگو', 'نیست', 'راضی', 'پیشنهاد', 'نمی‌کنم')])
print("Ba hazf nafi (Khatarnak):", [w for w in dropped_neg if w in ('خوب', 'نبود', 'پاسخگو', 'نیست', 'راضی', 'پیشنهاد', 'نمی‌کنم')])

Top kalame haye tashkhis dade shode tavasot ML Stopword Classifier:
  {'token': 'و', 'tf': 0.0737, 'idf': 0.2335, 'length': 1, 'stopword_prob': 0.9047}
  {'token': 'در', 'tf': 0.0184, 'idf': 0.3877, 'length': 2, 'stopword_prob': 0.6658}
  {'token': '،', 'tf': 0.0292, 'idf': 0.7931, 'length': 1, 'stopword_prob': 0.6604}
  {'token': 'که', 'tf': 0.0092, 'idf': 0.3877, 'length': 2, 'stopword_prob': 0.6279}
  {'token': 'به', 'tf': 0.0169, 'idf': 0.57, 'length': 2, 'stopword_prob': 0.6002}

--- Moghayese 5 Ravesh Stopwords rooye Doc #6 ---
dictionary       -> Baghi-mande: 47 | Hazf shode: ['و', 'بر', 'طبق', 'خود', 'را', 'به', 'کرد', 'و', 'که', 'پس', 'از', 'است', 'در', 'در', 'و', 'شده', 'و', 'در', 'داشت']
statistical_idf  -> Baghi-mande: 58 | Hazf shode: ['و', 'و', 'که', 'در', 'در', 'و', 'و', 'در']
domain_specific  -> Baghi-mande: 62 | Hazf shode: ['ماده', 'قانون', 'شاکی', 'دادگاه']
ml_classifier    -> Baghi-mande: 56 | Hazf shode: ['و', 'را', 'به', 'و', 'که', 'در', 'در', 'و', 'و', 'در']
hybr

## گام ۶: مقایسه ریشه‌یابی (`Stemming` در برابر `Lemmatization`) و کاهش اندازه واژگان

In [6]:
morph_doc_fa = raw_docs[4]
_, toks_fa, _ = tok_engine.tokenize(normalizer.normalize(cleaner.clean(morph_doc_fa)[0])[0])
filt_fa, _ = sw_base.filter_tokens(toks_fa)

cfg_lemma = copy.deepcopy(PIPELINE_CONFIG)
cfg_lemma["morphology"]["mode"] = "lemma"
lemmas_fa, aud_fa = MorphologyProcessor(cfg_lemma).process(filt_fa)

print(f"Tedad Type haye yekta ghabl az Lemmatization: {len(set(filt_fa))}")
print(f"Tedad Type haye yekta bad az Lemmatization  : {len(set(lemmas_fa))}")
print("Jadval negasht rishe haye Farsi:")
for m in aud_fa["mappings"]:
    print(f"  {m['original']}  --->  {m['root']}")

filt_en, _ = sw_base.filter_tokens(tok_engine.tokenize(n_en)[1])
cfg_stem = copy.deepcopy(PIPELINE_CONFIG)
cfg_stem["morphology"]["mode"] = "stem"
cfg_stem["morphology"]["en_stemmer"] = "porter"
stems_en, _ = MorphologyProcessor(cfg_stem).process(filt_en)
lemmas_en, _ = MorphologyProcessor(cfg_lemma).process(filt_en)

print("\nMoghayese English (Original vs Porter Stem vs Lemma):")
for orig, st, lm in list(zip(filt_en, stems_en, lemmas_en))[:18]:
    if orig != st or orig != lm:
        print(f"  {orig:<15} | Stem: {st:<15} | Lemma: {lm}")

Tedad Type haye yekta ghabl az Lemmatization: 81
Tedad Type haye yekta bad az Lemmatization  : 80
Jadval negasht rishe haye Farsi:
  چالش‌های  --->  چالش
  ده‌ها  --->  ده
  می‌روم  --->  رفت
  می‌روم  --->  رفت
  رفتم  --->  رفت
  می‌خواهم  --->  خواست
  می‌خواهم  --->  خواست
  نوشتم  --->  نوشت
  می‌کنم  --->  کرد
  می‌کنم  --->  کرد
  می‌خوانم  --->  خواند
  کتاب‌ها  --->  کتاب
  کتاب‌ها  --->  کتاب
  دانشجویان  --->  دانشجو
  مقالات  --->  مقاله
  بزرگ‌ترین  --->  بزرگ
  بزرگ‌ترین  --->  بزرگ
  بردار‌های  --->  بردار
  words  --->  word
  ویژگی‌ها  --->  ویژگی
  ایست‌واژه‌های  --->  ایست‌واژه
  می‌گیرند  --->  گرفت

Moghayese English (Original vs Porter Stem vs Lemma):
  studying        | Stem: studi           | Lemma: study
  multilingual    | Stem: multilingu      | Lemma: multilingual
  corpora         | Stem: corpora         | Lemma: corpus
  running         | Stem: run             | Lemma: run


## گام ۷: نمایش برداری (`Bag-of-Words` در برابر `TF-IDF`) و موتور جستجوی کسینوسی (با پشتیبانی از غلط املایی)

In [7]:
full_pipeline = MultilingualNLPPipeline(PIPELINE_CONFIG)
query_text = "پردازش زبان طبیعی و مدل‌های زبانی و بازیابی دانش"
full_report = full_pipeline.analyze_corpus(raw_docs, search_query=query_text)

vr = full_report["text_representation_and_similarity"]
print("=== MATRIX & SPARSITY ANALYTICS ===")
print(f"Shape: {vr['matrix_shape']} | Sparsity: {vr['sparsity_percent']}% | Memory Saved: {vr['memory_comparison']['saved_percent']}%")

sr = full_report["mini_search_engine_demo"]
print("\n=== QUERY TOKEN DENSE VECTORS (8D) ===")
for item in sr["query_token_dense_vectors"][:5]:
    print(f"  Token '{item['token']}' -> {item['dense_vector']}")
print("  Pooled Query Vector ->", sr["query_pooled_dense_vector"])

print(f"\n=== SEARCH RESULTS FOR EXACT QUERY: '{query_text}' ===")
for match in sr["top_matches"][:3]:
    print(f"Rank {match['rank']} | Doc #{match['doc_id']} | Type: {match['match_type']} | Score: {match['score']} | Terms: {match['matched_terms']}")

typo_report = full_pipeline.analyze_corpus(raw_docs, search_query="نوروپاطی")
print("\n=== SEARCH RESULTS FOR TYPO QUERY: 'نوروپاطی' ===")
for match in typo_report["mini_search_engine_demo"]["top_matches"][:2]:
    print(f"Rank {match['rank']} | Doc #{match['doc_id']} | Type: {match['match_type']} | Score: {match['score']} | Matched: {match['matched_terms']}")

=== MATRIX & SPARSITY ANALYTICS ===
Shape: [7, 797] | Sparsity: 84.28% | Memory Saved: 76.4%

=== QUERY TOKEN DENSE VECTORS (8D) ===
  Token 'پردازش' -> [0.206, 0.598, 0.15, -0.069, 0.592, -0.435, 0.15, 0.104]
  Token 'زبان' -> [0.586, 0.431, 0.233, 0.222, 0.062, -0.265, 0.356, -0.409]
  Token 'طبیعی' -> [0.249, -0.179, 0.533, -0.147, 0.514, -0.193, -0.142, -0.528]
  Token 'مدل' -> [0.431, -0.053, 0.528, -0.371, -0.07, -0.52, -0.346, -0.019]
  Token 'زبانی' -> [0.683, 0.159, 0.133, -0.142, 0.099, -0.631, 0.056, -0.245]
  Pooled Query Vector -> [0.388, 0.047, 0.174, -0.148, 0.08, -0.317, -0.116, -0.124]

=== SEARCH RESULTS FOR EXACT QUERY: 'پردازش زبان طبیعی و مدل‌های زبانی و بازیابی دانش' ===
Rank 1 | Doc #1 | Type: exact_tfidf_cosine | Score: 0.1828 | Terms: ['بازیابی', 'زبان', 'زبان طبیعی', 'طبیعی', 'پردازش']
Rank 2 | Doc #3 | Type: exact_tfidf_cosine | Score: 0.1097 | Terms: ['بازیابی', 'بازیابی دانش', 'دانش', 'پردازش']
Rank 3 | Doc #2 | Type: exact_tfidf_cosine | Score: 0.1073 | Te

## گام ۸: مقایسه نهایی ۶ خانواده مدل کلاسیک و مدرن (Rule-Based، Naive Bayes، Logistic Regression، SVM، Random Forest و Transformer)

In [8]:
bench_data = full_report["models_benchmark_and_comparison"]

print("=== 1. BENCHMARK METRICS ON 48-SENTENCE CHALLENGING DATASET (32 TRAIN / 16 TEST) ===")
for model_name, m in bench_data["benchmark_metrics"].items():
    print(f"{model_name:<28} | Acc: {m['Accuracy']*100:5.1f}% | Prec: {m['Precision']*100:5.1f}% | Rec: {m['Recall']*100:5.1f}% | F1: {m['F1_Score']*100:5.1f}% | CM: {m['Confusion_Matrix']}")

print("\n=== 2. LIVE PREDICTION COMPARISON ON COMPLEX NEGATIVE REVIEW (DOC #2) ===")
preds = bench_data["single_text_predictions"]
for k, v in preds.items():
    if k not in ("input_text", "processed_tokens"):
        print(f"  {k:<28} -> {v}")

=== 1. BENCHMARK METRICS ON 48-SENTENCE CHALLENGING DATASET (32 TRAIN / 16 TEST) ===
1_Rule_Based_Lexicon         | Acc: 100.0% | Prec: 100.0% | Rec: 100.0% | F1: 100.0% | CM: {'TP': 8, 'TN': 8, 'FP': 0, 'FN': 0}
2_Multinomial_Naive_Bayes    | Acc:  81.2% | Prec:  85.7% | Rec:  75.0% | F1:  80.0% | CM: {'TP': 6, 'TN': 7, 'FP': 1, 'FN': 2}
3_Logistic_Regression        | Acc:  87.5% | Prec: 100.0% | Rec:  75.0% | F1:  85.7% | CM: {'TP': 6, 'TN': 8, 'FP': 0, 'FN': 2}
4_Linear_SVM                 | Acc:  56.2% | Prec: 100.0% | Rec:  12.5% | F1:  22.2% | CM: {'TP': 1, 'TN': 8, 'FP': 0, 'FN': 7}
5_Random_Forest_Ensemble     | Acc:  50.0% | Prec:   0.0% | Rec:   0.0% | F1:   0.0% | CM: {'TP': 0, 'TN': 8, 'FP': 0, 'FN': 8}
6_Transformer_Attention      | Acc: 100.0% | Prec: 100.0% | Rec: 100.0% | F1: 100.0% | CM: {'TP': 8, 'TN': 8, 'FP': 0, 'FN': 0}

=== 2. LIVE PREDICTION COMPARISON ON COMPLEX NEGATIVE REVIEW (DOC #2) ===
  1_Rule_Based_Lexicon         -> {'prediction': 'Negative / Critical (م